# Export, reload, remove, and upgrade packages

This NB-007 workflow saves an anchored composition, reloads it from local artifacts, and validates explicit unload, reinsertion, upgrade, and rollback synchronization. The smoke fixture is deterministic and performs no network access.

## Resolve configuration

Keep the external anchor reference, package versions, converted layers, and exact comparison tolerances in one resolved mapping.

In [ ]:
import platform
from pathlib import Path
from tempfile import TemporaryDirectory

import safetensors
import torch
import transformers
from transformers import Qwen3Config, Qwen3ForCausalLM

from anchored_moe import (
    EXPERT_PAYLOAD_BINDING_FORMAT_VERSION,
    AffineRouterHead,
    AnchoredMoEForCausalLM,
    AppendOnlyRouterHeads,
    ExpertPackage,
    ExpertPackageManifest,
    FullMLPExpert,
    RoutedMLP,
)
from anchored_moe.evaluation import assert_tensors_equivalent
from anchored_moe.registry import PACKAGE_FORMAT_VERSION
from anchored_moe.utils import sha256_file, sha256_json

RESOLVED_CONFIG = {
    "feature_id": "NB-007",
    "seed": 13,
    "base_model_id": "local-random-qwen3",
    "base_model_revision": "synthetic-seed-13",
    "converted_layers": [2, 3],
    "specialization": "code",
    "expert_id": "code:0",
    "package_versions": [
        {"version": "1.0.0", "expert_parameter_value": 0.0},
        {"version": "2.0.0", "expert_parameter_value": 0.05},
    ],
    "input_ids": [[1, 7, 11, 3]],
    "atol": 0.0,
    "rtol": 0.0,
}
RESOLVED_CONFIG

## Load the immutable anchor and prepare package versions

The composition checkpoint references this separately saved anchor. Both immutable package versions target the same anchor and router-feature compatibility identity.

In [ ]:
torch.manual_seed(RESOLVED_CONFIG["seed"])
anchor = Qwen3ForCausalLM(
    Qwen3Config(
        vocab_size=64,
        hidden_size=16,
        intermediate_size=32,
        num_hidden_layers=4,
        num_attention_heads=4,
        num_key_value_heads=2,
        head_dim=4,
        max_position_embeddings=32,
    )
).eval()
input_ids = torch.tensor(RESOLVED_CONFIG["input_ids"])
with torch.no_grad():
    dense_logits = anchor(input_ids=input_ids).logits

temporary_workspace = TemporaryDirectory(prefix="anchored-moe-nb007-")
workspace = Path(temporary_workspace.name)
anchor_checkpoint = workspace / "anchor"
anchor.save_pretrained(anchor_checkpoint)
model = AnchoredMoEForCausalLM.from_pretrained(
    anchor_checkpoint,
    converted_layers=RESOLVED_CONFIG["converted_layers"],
    local_files_only=True,
).eval()
assert all(not parameter.requires_grad for parameter in model.parameters())

In [ ]:
packages = {}
for package_specification in RESOLVED_CONFIG["package_versions"]:
    version = package_specification["version"]
    package_root = workspace / f"code-{version}"
    package_root.mkdir()
    weight_files = []
    payload_bindings = []
    package_hashes = {}
    for layer_index in RESOLVED_CONFIG["converted_layers"]:
        wrapper = model.anchor_model.model.layers[layer_index].mlp
        assert isinstance(wrapper, RoutedMLP)
        expert = FullMLPExpert.from_anchor(wrapper.anchor_mlp, trainable=False)
        with torch.no_grad():
            for parameter in expert.parameters():
                parameter.fill_(package_specification["expert_parameter_value"])
        relative_path = f"experts/layer-{layer_index}-code-0.safetensors"
        payload_path = package_root / relative_path
        payload_path.parent.mkdir(exist_ok=True)
        expert.save_safetensors(payload_path)
        weight_files.append(relative_path)
        payload_bindings.append({
            "layer_index": layer_index,
            "expert_id": RESOLVED_CONFIG["expert_id"],
            "weight_file": relative_path,
        })
        package_hashes[relative_path] = sha256_file(payload_path)

    router = AppendOnlyRouterHeads(model.anchor_model.config.hidden_size)
    for layer_index in RESOLVED_CONFIG["converted_layers"]:
        head = AffineRouterHead(
            model.anchor_model.config.hidden_size,
            layer_index=layer_index,
            expert_id=RESOLVED_CONFIG["expert_id"],
            trainable=False,
        )
        with torch.no_grad():
            head.linear.bias.fill_(1.0)
        router.append(head)
    router_relative_path = "routers/affine-heads.safetensors"
    router_path = package_root / router_relative_path
    router_path.parent.mkdir()
    router.save_safetensors(router_path)
    package_hashes[router_relative_path] = sha256_file(router_path)

    manifest = ExpertPackageManifest(
        package_format_version=PACKAGE_FORMAT_VERSION,
        specialization_name=RESOLVED_CONFIG["specialization"],
        specialization_version=version,
        package_structure="single_expert",
        base_model_repository=RESOLVED_CONFIG["base_model_id"],
        base_model_revision=RESOLVED_CONFIG["base_model_revision"],
        base_config_sha256=sha256_json(model.anchor_model.config.to_dict()),
        base_weight_sha256={"model.safetensors": "0" * 64},
        tokenizer_repository=RESOLVED_CONFIG["base_model_id"],
        tokenizer_revision=RESOLVED_CONFIG["base_model_revision"],
        base_model_dtype="float32",
        converted_layers=RESOLVED_CONFIG["converted_layers"],
        expert_parameterization="full_mlp",
        expert_parameterization_config={
            "initialization": "anchor_copy",
            "payload_binding_format_version": EXPERT_PAYLOAD_BINDING_FORMAT_VERSION,
            "payload_bindings": payload_bindings,
        },
        expert_count=1,
        weight_dtype="float32",
        expert_weight_files=weight_files,
        router_architecture="independent_affine_heads",
        routing_policies_supported=["anchored_top1"],
        router_feature_specification={
            "feature_type": "identity",
            "input": "pre_mlp_hidden_state",
            "output_size": model.anchor_model.config.hidden_size,
            "version": 1,
        },
        router_files=[router_relative_path],
        runtime_versions={
            "accelerate": "not-installed",
            "kernels": "torch-default-cpu",
            "peft": "not-installed",
            "python": platform.python_version(),
            "safetensors": safetensors.__version__,
            "torch": torch.__version__,
            "transformers": transformers.__version__,
        },
        training_dataset={
            "dataset_id": "synthetic-local-fixture",
            "revision": version,
            "split": "train",
            "filters": [],
        },
        package_hashes=package_hashes,
    )
    manifest.save_json(package_root / "manifest.json")
    packages[version] = ExpertPackage.from_directory(package_root)

code_v1 = packages["1.0.0"]
code_v2 = packages["2.0.0"]

## Run framework operations

Install and export version 1, reload the composition, then remove, reinsert, upgrade, and roll back. Each registry mutation is observed before and after explicit synchronization.

In [ ]:
model.install(code_v1)
installed = model.synchronize_registry()
with torch.no_grad():
    v1_logits = model(input_ids=input_ids).logits
assert not torch.equal(v1_logits, dense_logits)

composition_checkpoint = workspace / "composition-v1"
model.save_pretrained(composition_checkpoint)
reloaded = AnchoredMoEForCausalLM.from_pretrained(
    composition_checkpoint,
    local_files_only=True,
).eval()
with torch.no_grad():
    reloaded_v1_logits = reloaded(input_ids=input_ids).logits
reload_result = assert_tensors_equivalent(
    reloaded_v1_logits,
    v1_logits,
    name="reloaded v1 composition logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)
assert reloaded.synchronized_registry_sha256 == installed.registry_sha256

reloaded.unload(code_v1.identity)
with torch.no_grad():
    stale_unload_logits = reloaded(input_ids=input_ids).logits
assert_tensors_equivalent(
    stale_unload_logits,
    v1_logits,
    name="unsynchronized unload logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)
removed = reloaded.synchronize_registry()
with torch.no_grad():
    removed_logits = reloaded(input_ids=input_ids).logits
removed_result = assert_tensors_equivalent(
    removed_logits,
    dense_logits,
    name="removed anchor logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)

reloaded.install(ExpertPackage.from_directory(code_v1.package_directory))
with torch.no_grad():
    stale_reinsert_logits = reloaded(input_ids=input_ids).logits
assert_tensors_equivalent(
    stale_reinsert_logits,
    dense_logits,
    name="unsynchronized reinsertion logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)
reinserted = reloaded.synchronize_registry()
with torch.no_grad():
    reinserted_logits = reloaded(input_ids=input_ids).logits
reinserted_result = assert_tensors_equivalent(
    reinserted_logits,
    v1_logits,
    name="reinserted v1 logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)

upgrade = reloaded.upgrade(code_v1.identity, code_v2)
with torch.no_grad():
    stale_upgrade_logits = reloaded(input_ids=input_ids).logits
assert_tensors_equivalent(
    stale_upgrade_logits,
    v1_logits,
    name="unsynchronized upgrade logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)
upgraded = reloaded.synchronize_registry()
with torch.no_grad():
    v2_logits = reloaded(input_ids=input_ids).logits
assert not torch.equal(v2_logits, v1_logits)

rollback = reloaded.rollback(upgrade, code_v1)
with torch.no_grad():
    stale_rollback_logits = reloaded(input_ids=input_ids).logits
assert_tensors_equivalent(
    stale_rollback_logits,
    v2_logits,
    name="unsynchronized rollback logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)
restored = reloaded.synchronize_registry()
with torch.no_grad():
    restored_logits = reloaded(input_ids=input_ids).logits
rollback_result = assert_tensors_equivalent(
    restored_logits,
    v1_logits,
    name="rolled-back v1 logits",
    atol=RESOLVED_CONFIG["atol"],
    rtol=RESOLVED_CONFIG["rtol"],
)

## Validate invariants and summarize

Confirm exact restoration, immutable registry transitions, complete parameter freezing, and the documented limitation that inactive history is not embedded in a composition checkpoint.

In [ ]:
trainable_parameter_count = sum(
    parameter.numel()
    for parameter in reloaded.parameters()
    if parameter.requires_grad
)
assert reload_result.exact
assert removed_result.exact
assert reinserted_result.exact
assert rollback_result.exact
assert removed.reset_layers == tuple(RESOLVED_CONFIG["converted_layers"])
assert reinserted.registry_sha256 == installed.registry_sha256
assert upgraded.registry_sha256 != installed.registry_sha256
assert rollback.after == upgrade.before
assert restored.registry_sha256 == installed.registry_sha256
assert trainable_parameter_count == 0

lifecycle_summary = {
    "feature_id": RESOLVED_CONFIG["feature_id"],
    "composition_exported": composition_checkpoint.is_dir(),
    "reload_exact": reload_result.exact,
    "removed_to_anchor_exact": removed_result.exact,
    "reinsertion_exact": reinserted_result.exact,
    "upgrade_changed_output": not torch.equal(v2_logits, v1_logits),
    "rollback_exact": rollback_result.exact,
    "registry_sha256_restored": restored.registry_sha256 == installed.registry_sha256,
    "explicit_synchronization_required": True,
    "trainable_parameter_count": trainable_parameter_count,
    "inactive_history_persisted": False,
}
print("NB-007 validation passed", lifecycle_summary)
temporary_workspace.cleanup()